In [1]:
from ska_sdp_instrumental_calibration.scheduler import UpstreamOutput
from ska_sdp_instrumental_calibration.stages import (
    load_data_stage, predict_vis_stage
)

from ska_sdp_instrumental_calibration.numpy_processors.solvers import Solver
from ska_sdp_instrumental_calibration.plot import PlotGaintableFrequency
from ska_sdp_instrumental_calibration.xarray_processors._utils import parse_antenna
from ska_sdp_instrumental_calibration.xarray_processors.solver import run_solver
from ska_sdp_instrumental_calibration.stages._utils import get_plots_path
from ska_sdp_instrumental_calibration.xarray_processors.delay import apply_delay, calculate_delay
from ska_sdp_instrumental_calibration.plot import plot_station_delays, PlotGaintableFrequency

import numpy as np
import xarray as xr
import dask
import matplotlib.pyplot as plt

In [2]:
import pandas as pd

def unstack_jones_coordinate(ref_gaintable, gaintable):
    stack_gains = gaintable.gain.data
    new_gain_data = ref_gaintable.gain.data.copy()
    new_gain_data[..., 0, 0] = stack_gains[..., 0]
    new_gain_data[..., 1, 1] = stack_gains[..., 1]

    new_gain = ref_gaintable.gain.copy()
    new_gain.data = new_gain_data

    return ref_gaintable.assign(
        {
            "gain": new_gain,
        }
    ).chunk(ref_gaintable.chunks)
    
    
def stack_jones_coordinate(gaintable):
    gaintable = gaintable.stack(Jones_Solutions=("receptor1", "receptor2"))

    polstrs = [
        f"J_{p1}{p2}".upper()
        for p1, p2 in gaintable["Jones_Solutions"].data
    ]
    gaintable = gaintable.drop_vars(['Jones_Solutions', 'receptor1', 'receptor2']).assign_coords({"Jones_Solutions": polstrs})

    return gaintable

In [3]:
def delay_calibration(
    upstream_output,
    output_path,
    refant = 0,
    niter = 200,
    tol  = 1e-06,
    export_gaintable = True,
):
    vis = upstream_output.vis
    modelvis = upstream_output.modelvis
    initialtable = upstream_output.gaintable
    prefix = upstream_output.ms_prefix

    refant = parse_antenna(refant, initialtable.configuration.names)
    solver = Solver.get_solver(refant=refant, niter=niter, tol=tol) 
    gaintables = []
    pols = ["XX", "YY"]

    for pol in pols:
        scalar_vis = vis.sel(polarisation=[pol])
        scalar_model_vis = modelvis.sel(polarisation=[pol])
        scalar_table = initialtable.sel(receptor1=[pol[0]], receptor2=[pol[0]])
        scalar_gaintable = run_solver(  # update
            vis=scalar_vis,
            modelvis=scalar_model_vis,
            gaintable=scalar_table,
            solver=solver,
        )
    
        gaintables.append(scalar_gaintable.compute())

    gaintable = unstack_jones_coordinate(
        initialtable, 
        xr.merge([stack_jones_coordinate(table) for table in gaintables])
    )
    
    delaytable = calculate_delay(gaintable, oversample=1)
    corrected_gaintable = apply_delay(gaintable, delaytable)
    
    prefix = upstream_output.ms_prefix
    path_prefix = get_plots_path(
            output_path, f"{prefix}/delay"
        )

    upstream_output.add_compute_tasks(
        plot_station_delays(
            delaytable,
            path_prefix,
        )
    )

    freq_plotter = PlotGaintableFrequency(
        path_prefix=path_prefix,
        refant=refant,
    )

    upstream_output.add_compute_tasks(
        *freq_plotter.plot(
                gaintable,
                figure_title="Delay",
                fixed_axis=False,
        )
    )

    freq_plotter = PlotGaintableFrequency(
        path_prefix=f"{path_prefix}_corrected",
        refant=refant,
    )
    
    upstream_output.add_compute_tasks(
        *freq_plotter.plot(
                corrected_gaintable,
                figure_title="Delay Corrected",
                fixed_axis=False,
        )
    )
    
    upstream_output["gaintable"] = gaintable
    upstream_output["refant"] = refant

    return upstream_output


In [4]:
input_ms = ["/data/SKA/inst_data/test_delay_visbility_240.ms"]
lsm_csv = "/data/SKA/inst_data/local_sky_model.csv"
output_dir = "./delay_cal_spike_output"

upstream_output = UpstreamOutput()

In [5]:
upstream_output = load_data_stage(
    upstream_output,
    output_dir,
    input_ms=input_ms,
    cache_directory="../../output_wide_band_data",
)

upstream_output = predict_vis_stage(
    upstream_output[0], output_dir, input_ms=input_ms, lsm_csv_path=lsm_csv
)

upstream_output = delay_calibration(
    upstream_output,
    output_dir
)




1|2026-06-19T07:38:32.676Z|INFO|upside-down|MainThread|_load_data|load_data.py#183||Reading cached visibilities from path ../../output_wide_band_data/test_delay_visbility_240.ms_fid0_ddid0
Successful readonly open of default-locked table /data/SKA/inst_data/test_delay_visbility_240.ms/FIELD: 9 columns, 1 rows
1|2026-06-19T07:38:32.737Z|INFO|upside-down|MainThread|__init__|local_sky_model.py#204||Generating GSM for predict with:
1|2026-06-19T07:38:32.737Z|INFO|upside-down|MainThread|__init__|local_sky_model.py#205|| - Search radius: 2.5 deg
1|2026-06-19T07:38:32.738Z|INFO|upside-down|MainThread|__init__|local_sky_model.py#206|| - Flux limit: 1.0 Jy
1|2026-06-19T07:38:32.738Z|INFO|upside-down|MainThread|__init__|local_sky_model.py#218|| - Catalogue file: /data/SKA/inst_data/local_sky_model.csv
1|2026-06-19T07:38:32.758Z|INFO|upside-down|MainThread|generate_lsm_from_csv|sky_model_reader.py#233||extracted 16 csv components
1|2026-06-19T07:38:32.758Z|INFO|upside-down|MainThread|__init__|loc

2026-06-19 07:38:33	SEVERE	MeasTable::dUTC(Double) (file /tmp/casacore-3.7.1/measures/Measures/MeasTable.cc, line 4290)	Leap second table TAI_UTC seems out-of-date.
2026-06-19 07:38:33	SEVERE	MeasTable::dUTC(Double) (file /tmp/casacore-3.7.1/measures/Measures/MeasTable.cc, line 4290)+	Until the table is updated (see the CASA documentation or your system admin),
2026-06-19 07:38:33	SEVERE	MeasTable::dUTC(Double) (file /tmp/casacore-3.7.1/measures/Measures/MeasTable.cc, line 4290)+	times and coordinates derived from UTC could be wrong by 1s or more.


1|2026-06-19T07:38:33.662Z|INFO|upside-down|ThreadPoolExecutor-0_17|__init__|beams.py#126||Setting beam normalisation for OSKAR data
1|2026-06-19T07:38:33.691Z|INFO|upside-down|ThreadPoolExecutor-0_0|__init__|beams.py#126||Setting beam normalisation for OSKAR data
1|2026-06-19T07:38:33.727Z|INFO|upside-down|ThreadPoolExecutor-0_8|__init__|beams.py#126||Setting beam normalisation for OSKAR data
1|2026-06-19T07:38:33.891Z|INFO|upside-down|ThreadPoolExecutor-0_13|__init__|beams.py#126||Setting beam normalisation for OSKAR data
1|2026-06-19T07:38:33.961Z|INFO|upside-down|ThreadPoolExecutor-0_22|__init__|beams.py#126||Setting beam normalisation for OSKAR data
1|2026-06-19T07:38:33.990Z|INFO|upside-down|ThreadPoolExecutor-0_2|__init__|beams.py#126||Setting beam normalisation for OSKAR data
1|2026-06-19T07:38:34.013Z|INFO|upside-down|ThreadPoolExecutor-0_20|__init__|beams.py#126||Setting beam normalisation for OSKAR data
1|2026-06-19T07:38:34.019Z|INFO|upside-down|ThreadPoolExecutor-0_18|__in

/home/justin/miniconda3/envs/inst/lib/python3.10/site-packages/xarray/core/common.py:181: ComplexWarning: Casting complex values to real discards the imaginary part
  return np.array(self.values, dtype=dtype, copy=copy)
/home/justin/miniconda3/envs/inst/lib/python3.10/site-packages/xarray/core/common.py:181: ComplexWarning: Casting complex values to real discards the imaginary part
  return np.array(self.values, dtype=dtype, copy=copy)


In [6]:
gaintable = upstream_output.gaintable.compute()
vis = upstream_output.vis.compute()

1|2026-06-19T07:38:58.822Z|INFO|upside-down|ThreadPoolExecutor-0_13|__init__|beams.py#126||Setting beam normalisation for OSKAR data
1|2026-06-19T07:38:58.839Z|INFO|upside-down|ThreadPoolExecutor-0_9|__init__|beams.py#126||Setting beam normalisation for OSKAR data
1|2026-06-19T07:38:58.880Z|INFO|upside-down|ThreadPoolExecutor-0_3|__init__|beams.py#126||Setting beam normalisation for OSKAR data
1|2026-06-19T07:38:58.917Z|INFO|upside-down|ThreadPoolExecutor-0_17|__init__|beams.py#126||Setting beam normalisation for OSKAR data
1|2026-06-19T07:38:58.948Z|INFO|upside-down|ThreadPoolExecutor-0_1|__init__|beams.py#126||Setting beam normalisation for OSKAR data
1|2026-06-19T07:38:58.994Z|INFO|upside-down|ThreadPoolExecutor-0_14|__init__|beams.py#126||Setting beam normalisation for OSKAR data
1|2026-06-19T07:38:59.021Z|INFO|upside-down|ThreadPoolExecutor-0_8|__init__|beams.py#126||Setting beam normalisation for OSKAR data
1|2026-06-19T07:38:59.041Z|INFO|upside-down|ThreadPoolExecutor-0_0|__init

In [7]:
dask.compute(*upstream_output.compute_tasks)

1|2026-06-19T07:38:59.444Z|INFO|upside-down|ThreadPoolExecutor-0_21|_plot_bandpass_terms|plot_gaintable.py#201||Gaintable plotting started for figure: Delay Leakage
1|2026-06-19T07:38:59.454Z|INFO|upside-down|ThreadPoolExecutor-0_6|_plot_bandpass_terms|plot_gaintable.py#201||Gaintable plotting started for figure: Delay Gain
1|2026-06-19T07:38:59.455Z|INFO|upside-down|ThreadPoolExecutor-0_22|_plot_bandpass_terms|plot_gaintable.py#201||Gaintable plotting started for figure: Delay Corrected Gain
1|2026-06-19T07:38:59.455Z|INFO|upside-down|ThreadPoolExecutor-0_13|_plot_bandpass_terms|plot_gaintable.py#201||Gaintable plotting started for figure: Delay Corrected Leakage


/home/justin/Documents/Projects/SKA/ska-sdp-instrumental-calibration/src/ska_sdp_instrumental_calibration/plot/plot_gaintable.py:487: FutureWarning: updating coordinate 'Jones_Solutions' with a PandasMultiIndex would leave the multi-index level coordinates ['receptor1', 'receptor2'] in an inconsistent state. This will raise an error in the future. Use `.drop_vars(['Jones_Solutions', 'receptor1', 'receptor2'])` before assigning new coordinate values.
  gaintable = gaintable.assign_coords({"Jones_Solutions": polstrs})
/home/justin/Documents/Projects/SKA/ska-sdp-instrumental-calibration/src/ska_sdp_instrumental_calibration/plot/plot_gaintable.py:487: FutureWarning: updating coordinate 'Jones_Solutions' with a PandasMultiIndex would leave the multi-index level coordinates ['receptor1', 'receptor2'] in an inconsistent state. This will raise an error in the future. Use `.drop_vars(['Jones_Solutions', 'receptor1', 'receptor2'])` before assigning new coordinate values.
  gaintable = gaintable.

1|2026-06-19T07:39:26.336Z|INFO|upside-down|ThreadPoolExecutor-0_6|_plot_bandpass_terms|plot_gaintable.py#253||Gaintable plots saved with prefix ./delay_cal_spike_output/plots/test_delay_visbility_240/delay.
1|2026-06-19T07:39:28.338Z|INFO|upside-down|ThreadPoolExecutor-0_13|_plot_bandpass_terms|plot_gaintable.py#253||Gaintable plots saved with prefix ./delay_cal_spike_output/plots/test_delay_visbility_240/delay_corrected.
1|2026-06-19T07:39:28.966Z|INFO|upside-down|ThreadPoolExecutor-0_22|_plot_bandpass_terms|plot_gaintable.py#253||Gaintable plots saved with prefix ./delay_cal_spike_output/plots/test_delay_visbility_240/delay_corrected.
1|2026-06-19T07:39:29.311Z|INFO|upside-down|ThreadPoolExecutor-0_21|_plot_bandpass_terms|plot_gaintable.py#253||Gaintable plots saved with prefix ./delay_cal_spike_output/plots/test_delay_visbility_240/delay.


(None, None, None, None, None)

In [ ]:
# %matplotlib inline
# Apply gaintable to visibility
# plot visibility
# Verify with Abhinav